In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [2]:
%pip install ucimlrepo


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

### Explore the Data

In [4]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

#### Check for duplicates

In [5]:
X.duplicated().sum()

np.int64(0)

In [6]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

#### Split the dataset

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((455, 30), (455,), (114, 30), (114,))

**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

### Define models, apply transformation, select features and evaluate the models

**Define Candidate Model Pipelines**

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Standardized Cross Validation Evaluation

In [9]:
import time
import numpy as np
import pandas as pd
import json
import joblib
from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEEDS = [11, 42, 67]
N_REPEATS = len(RANDOM_SEEDS)
CV_FOLDS = 5
ML_MODELS = {"LR": LR_pipeline, "SVC": SVC_pipeline, "RF": RF_pipeline, "GB": GB_pipeline, "XGB": XGB_pipeline}


#### Select top features using SelectKBest + GridCVSearch

In [10]:
def feature_search(model, X, cv):

    pipeline = Pipeline([
        ("selector", SelectKBest(score_func=f_classif)),
        ("classifier", clone(model))
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid={
            "selector__k": list(range(1, X.shape[1] + 1))
        },
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    return search

In [11]:

def evaluate_model(model_name, model, x, y, cv):

    fold_results = []
    selected_k = []
    y_true_all = []
    y_pred_all = []
    y_score_all = []

    start_time = time.perf_counter()
    fit_count = 0
    model_evaluations = 0

    y_array = np.asarray(y)

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(x, y_array), start=1
    ):

        X_train_fold = (
            x.iloc[train_idx] if hasattr(x, "iloc")
            else x[train_idx]
        )
        X_val_fold = (
            x.iloc[val_idx] if hasattr(x, "iloc")
            else x[val_idx]
        )

        y_train_fold = y_array[train_idx]
        y_val_fold = y_array[val_idx]

        # Inner CV chooses the number of features
        search = feature_search(
            model, X_train_fold, cv
        )

        search.fit(X_train_fold, y_train_fold)

        # Count model evaluations and CV fits
        n_candidates = len(search.cv_results_["params"])
        n_inner_folds = search.n_splits_

        model_evaluations += n_candidates

        # Inner CV fits + one final refit per outer fold
        fit_count += n_candidates * n_inner_folds + 1

        # Retrieve the best fitted pipeline
        best_pipeline = search.best_estimator_

        selected_k.append(
            search.best_params_["selector__k"]
        )

        # Predict on untouched outer validation fold
        y_pred = best_pipeline.predict(X_val_fold)

        if hasattr(best_pipeline, "predict_proba"):
            pos_idx = list(best_pipeline.classes_).index(1)
            y_score = best_pipeline.predict_proba(
                X_val_fold
            )[:, pos_idx]

        elif hasattr(best_pipeline, "decision_function"):
            y_score = best_pipeline.decision_function(
                X_val_fold
            )
            if best_pipeline.classes_[1] != 1:
                y_score = -y_score
        else:
            raise ValueError(
                f"{model_name} cannot produce AUC scores"
            )

        y_true_all.extend(y_val_fold)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

        fold_results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_val_fold, y_pred),
            "Precision": precision_score(
                y_val_fold, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_val_fold, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_val_fold, y_pred, zero_division=0
            ),
            "AUC": roc_auc_score(y_val_fold, y_score)
        })

    runtime = time.perf_counter() - start_time

    folds_df = pd.DataFrame(fold_results)

    metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

    means = folds_df[metrics].mean()
    stds = folds_df[metrics].std(ddof=1)

    cm = confusion_matrix(
        y_true_all, y_pred_all, labels=[0, 1]
    )

    return {
        "Model": model_name,

        "Accuracy": means["Accuracy"],
        "Accuracy_std": stds["Accuracy"],

        "Precision": means["Precision"],
        "Precision_std": stds["Precision"],

        "Recall": means["Recall"],
        "Recall_std": stds["Recall"],

        "F1": means["F1"],
        "F1_std": stds["F1"],

        "AUC": means["AUC"],
        "AUC_std": stds["AUC"],

        "Selected_k_per_fold": selected_k,
        "Runtime_seconds": runtime,
        "Model_evaluations": model_evaluations,
        "CV_fits": fit_count,
        "Confusion_matrix": cm,
        "Fold_results": fold_results
    }


In [12]:
# Three independently seeded repeated nested CV runs per pipeline. X_test is not accessed.
repeated_results = []
for repeat, seed in enumerate(RANDOM_SEEDS, start=1):
    outer_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    for model_name, model in ML_MODELS.items():
        result = evaluate_model(model_name, model, X_train, y_train, outer_cv)
        result["Repeat"] = repeat
        result["Seed"] = seed
        result["CV_Folds"] = CV_FOLDS
        result["Development_CV_F1_Mean"] = result["F1"]
        result["Development_CV_F1_SD"] = result["F1_std"]
        result["Development_CV_Runtime_Seconds"] = result["Runtime_seconds"]
        repeated_results.append(result)

repeated_results_df = pd.DataFrame(repeated_results)


In [13]:
# Summarize each CV metric across the three repeated runs for every pipeline.
metric_names = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
summary_aggregations = {}
for metric in metric_names:
    summary_aggregations[f"Development_CV_{metric}_Mean"] = (metric, "mean")
    summary_aggregations[f"Development_CV_{metric}_SD"] = (metric, "std")
summary_aggregations["Development_CV_Runtime_Seconds_Mean"] = ("Development_CV_Runtime_Seconds", "mean")
summary_aggregations["Development_CV_Runtime_Seconds_SD"] = ("Development_CV_Runtime_Seconds", "std")
summary_df = (repeated_results_df.groupby("Model", as_index=False)
    .agg(**summary_aggregations)
    .sort_values("Development_CV_F1_Mean", ascending=False).reset_index(drop=True))
display(summary_df.round(4))


,Model,Development_CV_Accuracy_Mean,Development_CV_Accuracy_SD,Development_CV_Precision_Mean,Development_CV_Precision_SD,Development_CV_Recall_Mean,Development_CV_Recall_SD,Development_CV_F1_Mean,Development_CV_F1_SD,Development_CV_AUC_Mean,Development_CV_AUC_SD,Development_CV_Runtime_Seconds_Mean,Development_CV_Runtime_Seconds_SD
0,XGB,0.9663,0.0046,0.9632,0.0069,0.9471,0.0102,0.9543,0.0067,0.9915,0.0029,51.2890,1.2056
1,SVC,0.9656,0.0046,0.9604,0.0147,0.9490,0.0034,0.9540,0.0053,0.9939,0.0028,3.6783,0.4860
2,LR,0.9656,0.0063,0.9650,0.0102,0.9431,0.0068,0.9534,0.0082,0.9946,0.0015,6.2961,4.7655
3,RF,0.9546,0.0089,0.9505,0.0125,0.9294,0.0102,0.9386,0.0118,0.9870,0.0028,28.6412,1.6559
4,GB,0.9524,0.0034,0.9513,0.0070,0.9216,0.0034,0.9351,0.0044,0.9886,0.0025,28.5575,0.6743


In [14]:
best_model_name = summary_df.iloc[0]["Model"]
print("Best conventional classifier:", best_model_name)
print(f"Mean repeated CV F1: {summary_df.iloc[0]['Development_CV_F1_Mean']:.4f}")
print(f"Between-repeat F1 SD: {summary_df.iloc[0]['Development_CV_F1_SD']:.4f}")


Best conventional classifier: XGB
Mean repeated CV F1: 0.9543
Between-repeat F1 SD: 0.0067


In [15]:
# Final feature selection and model fit use development data only. No test scoring is performed.
final_search = feature_search(ML_MODELS[best_model_name], X_train,
    StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=101))
final_fit_start = time.perf_counter()
final_fit_cpu_start = time.process_time()
final_search.fit(X_train, y_train)
final_development_fit_seconds = time.perf_counter() - final_fit_start
final_fit_cpu_seconds = time.process_time() - final_fit_cpu_start
final_model = final_search.best_estimator_
final_k = final_search.best_params_["selector__k"]
selected_features = X_train.columns[final_model.named_steps["selector"].get_support()].tolist()

folder_name = "eval - feature selection"
output_dir = Path.cwd() if Path.cwd().name == folder_name else Path.cwd() / folder_name
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "conventional_frozen_model.joblib"
model_save_start = time.perf_counter()
joblib.dump(final_model, model_path)
model_save_seconds = time.perf_counter() - model_save_start
model_load_start = time.perf_counter()
loaded_best_pipeline = joblib.load(model_path)
model_load_seconds = time.perf_counter() - model_load_start
final_workflow_wall_seconds = final_development_fit_seconds + model_save_seconds + model_load_seconds
final_workflow_cpu_seconds = time.process_time() - final_fit_cpu_start

best_record = {
    "Best_Model": best_model_name,
    "Random_Seeds": RANDOM_SEEDS,
    "Final_Selected_Feature_Count": int(final_k),
    "Selected_Features": selected_features,
    "CV_Folds": CV_FOLDS,
    "Development_CV_F1_Mean": float(summary_df.iloc[0]["Development_CV_F1_Mean"]),
    "Development_CV_F1_SD": float(summary_df.iloc[0]["Development_CV_F1_SD"]),
    "Development_CV_Runtime_Seconds": float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_Mean"]),
    "Agent_Runtime_Seconds": None,
    "Final_Development_Fit_Runtime_Seconds": final_development_fit_seconds,
    "Model_Save_Runtime_Seconds": model_save_seconds,
    "Model_Load_Runtime_Seconds": model_load_seconds,
    "Final_Test_Scoring_Wall_Seconds": None,
    "Final_Test_Scoring_CPU_Seconds": None,
    "Final_Workflow_Wall_Seconds": final_workflow_wall_seconds,
    "Final_Workflow_CPU_Seconds": final_workflow_cpu_seconds,
    "Test_Scored": False,
    "Saved_Model_Path": str(model_path)
}
for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    best_record[f"Development_CV_{metric}_Mean"] = float(summary_df.iloc[0][f"Development_CV_{metric}_Mean"])
    best_record[f"Development_CV_{metric}_SD"] = float(summary_df.iloc[0][f"Development_CV_{metric}_SD"])
best_record["Development_CV_Runtime_Seconds_Mean"] = float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_Mean"])
best_record["Development_CV_Runtime_Seconds_SD"] = float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_SD"])
# Write reproducibility artifacts matching the useful outputs from the agentic workflow.
for seed in RANDOM_SEEDS:
    repeated_results_df.loc[repeated_results_df["Seed"] == seed].to_csv(
        output_dir / f"conventional_seed_{seed}_runs.csv", index=False)

seed_winners = (repeated_results_df.sort_values("Development_CV_F1_Mean", ascending=False)
    .groupby("Seed", as_index=False).first()[["Seed", "Model", "Development_CV_F1_Mean"]]
    .rename(columns={"Model": "Selected_Model", "Development_CV_F1_Mean": "Selected_Model_CV_F1"}))
seed_winners.to_csv(output_dir / "conventional_seed_selections.csv", index=False)
decision_consistency = (seed_winners.groupby("Selected_Model", as_index=False)
    .agg(Seed_Count=("Seed", "count"), Seeds=("Seed", lambda values: list(values)))
    .sort_values("Seed_Count", ascending=False))
decision_consistency.to_csv(output_dir / "decision_consistency.csv", index=False)

split_rows = pd.concat([
    pd.DataFrame({"Dataset_Row_Index": X_train.index.to_numpy(), "Partition": "development", "Target": y_train.to_numpy()}),
    pd.DataFrame({"Dataset_Row_Index": X_test.index.to_numpy(), "Partition": "test", "Target": y_test.to_numpy()})
], ignore_index=True)
split_rows.to_csv(output_dir / "split_manifest.csv", index=False)
partition_manifest = {
    "Split_Random_State": 42,
    "Test_Size_Fraction": 0.2,
    "Stratified": True,
    "Development_Row_Count": int(len(X_train)),
    "Test_Row_Count": int(len(X_test)),
    "Development_Target_Counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
    "Test_Target_Counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()}
}
(output_dir / "data_partition_manifest.json").write_text(json.dumps(partition_manifest, indent=2), encoding="utf-8")
run_metadata = {
    "Workflow": "conventional_pipeline_evaluation",
    "Random_Seeds": RANDOM_SEEDS,
    "CV_Folds": CV_FOLDS,
    "Repeated_Runs_Per_Pipeline": len(RANDOM_SEEDS),
    "Split_Random_State": 42,
    "Selection_Metric": "mean development CV F1 across seeds",
    "Test_Scored": False
}
(output_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
summary_df.to_csv(output_dir / "all_pipelines_summary.csv", index=False)
summary_df.to_csv(output_dir / "conventional_summary.csv", index=False)
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "development_fit_complete_test_pending",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Selection_Rule": run_metadata["Selection_Metric"],
    "Random_Seeds": RANDOM_SEEDS,
    "Split_Manifest": str(output_dir / "split_manifest.csv"),
    "Run_Metadata": str(output_dir / "run_metadata.json")
}, indent=2), encoding="utf-8")

repeated_results_df.to_csv(output_dir / "conventional_repeated_runs.csv", index=False)
summary_df.to_csv(output_dir / "conventional_pipeline_summary.csv", index=False)
(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Best model:", best_model_name)
print("Selected features:", selected_features)
print("Saved model and development-only record:", model_path)
print("Test set was not scored.")


Best model: XGB
Selected features: ['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 'concave_points1', 'symmetry1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 'compactness2', 'concavity2', 'concave_points2', 'fractal_dimension2', 'radius3', 'texture3', 'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 'fractal_dimension3']
Saved model and development-only record: c:\Users\ogeuc\Downloads\Conventional\eval - feature selection\conventional_frozen_model.joblib
Test set was not scored.


### Final Development Fit and Saved Model (Test Scoring Pending)


### Final Test Set Evaluation (Run Only After Confirmation)

This cell loads the saved best pipeline and evaluates it on the held-out test set. It is intentionally left unexecuted.


In [16]:
# Run this cell only when ready to evaluate the held-out test set.
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

loaded_best_pipeline = joblib.load(model_path)
test_wall_start = time.perf_counter()
test_cpu_start = time.process_time()
y_test_pred = loaded_best_pipeline.predict(X_test)
if hasattr(loaded_best_pipeline, "predict_proba"):
    class_index = list(loaded_best_pipeline.classes_).index(1)
    y_test_score = loaded_best_pipeline.predict_proba(X_test)[:, class_index]
else:
    y_test_score = loaded_best_pipeline.decision_function(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_auc = roc_auc_score(y_test, y_test_score)
test_scoring_cpu_seconds = time.process_time() - test_cpu_start
test_scoring_seconds = time.perf_counter() - test_wall_start

best_record.update({
    "Final_Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Final_Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Test_Scored": True,
    "Final_Test_Accuracy": test_accuracy,
    "Final_Test_Precision": test_precision,
    "Final_Test_Recall": test_recall,
    "Final_Test_F1": test_f1,
    "Final_Test_ROC_AUC": test_auc,
    "Final_Test_Confusion_Matrix": confusion_matrix(y_test, y_test_pred).tolist()
})
test_metrics = {
    "Model": best_model_name,
    "Test_Accuracy": test_accuracy,
    "Test_Precision": test_precision,
    "Test_Recall": test_recall,
    "Test_F1": test_f1,
    "Test_ROC_AUC": test_auc,
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds
}
pd.DataFrame([test_metrics]).to_csv(output_dir / "conventional_final_test.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "test_scoring_complete",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Test_Metrics_Path": str(output_dir / "conventional_final_test.csv"),
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Random_Seeds": RANDOM_SEEDS
}, indent=2), encoding="utf-8")
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)
(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected pipeline:", best_model_name)
print("Selected feature count:", final_k)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification report:\n", classification_report(y_test, y_test_pred, zero_division=0))


Selected pipeline: XGB
Selected feature count: 28
Test accuracy: 0.9649
Test precision: 1.0000
Test recall: 0.9048
Test F1: 0.9500
Test ROC AUC: 0.9907
Confusion matrix:
 [[72  0]
 [ 4 38]]

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

